In [18]:
pip install pandas scikit-learn werkzeug sqlalchemy flask_admin flask_migrate flask_rq2 flask_compress dnachisel openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)
Note: you may need to restart the kernel to use updated packages.


In [155]:
import time
from io import BytesIO
from datetime import datetime, UTC, timedelta
import traceback
import json
import io
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from werkzeug.exceptions import BadRequest
from pathlib import Path
import joblib
import matplotlib.pyplot

#from app.helpers.fold_storage_manager import FoldStorageManager
from app.helpers.sequence_util import (
    get_measured_and_unmeasured_mutant_seq_ids,
    get_loci_set,
    process_and_validate_evolve_input_files,
)




def train_model(wt_aa_seq,raw_activity_df,raw_embedding_df,evolve_directory):
    activity_df, embedding_df = process_and_validate_evolve_input_files(
                wt_aa_seq, raw_activity_df, raw_embedding_df
            )
    measured_mutants, unmeasured_mutants = (
                get_measured_and_unmeasured_mutant_seq_ids(activity_df, embedding_df)
            )
    X_train = np.vstack(
                [json.loads(x) for x in embedding_df.loc[activity_df.index].embedding]
            )
    y_train = activity_df.activity.to_numpy()
    model = RandomForestRegressor(
        n_estimators=100,
        criterion="friedman_mse",
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        min_weight_fraction_leaf=0.0,
        max_features=1.0,
        max_leaf_nodes=None,
        min_impurity_decrease=0.0,
        bootstrap=True,
        oob_score=False,
        n_jobs=None,
        random_state=1,
        verbose=0,
        warm_start=False,
        ccp_alpha=0.0,
        max_samples=None,
    )
    model.fit(X_train, y_train)
    try:
        all_mutants_embedding_array = np.vstack(
            [
                json.loads(x)
                for x in embedding_df.loc[
                    measured_mutants + unmeasured_mutants
                ].embedding
            ]
        )
        print(all_mutants_embedding_array.shape)
        y_all_pred = model.predict(all_mutants_embedding_array)
        predicted_activity_df = pd.DataFrame(
            {
                "seq_id": measured_mutants + unmeasured_mutants,
                "predicted_activity": y_all_pred,
            }
        )
        predicted_activity_df.index = predicted_activity_df.seq_id
        predicted_activity_df["relevant_measured_mutants"] = (
            predicted_activity_df.seq_id.apply(
                lambda seq_id: " ".join(
                    [
                        m
                        for m in measured_mutants
                        if get_loci_set(m) & get_loci_set(seq_id)
                    ]
                )
            )
        )
        predicted_activity_df["actual_activity"] = predicted_activity_df.join(
            activity_df.groupby(level=0).activity.mean(), how="left"
        ).activity
        predicted_activity_df = predicted_activity_df.sort_values(
            "predicted_activity", ascending=False
        )
    except Exception as e:
        print(f"Failed to predict activities: {e}")
        raise
    predicted_activity_df.reset_index(drop=True,inplace=True)
    #predicted_activity_csv_path = evolve_directory / f"Round_{round_num}_predicted_activity.csv"
    #print(f"Storing predicted activities in {predicted_activity_csv_path}")
    #try:
    #    predicted_activity_df.to_csv(predicted_activity_csv_path, index=False)
    #except Exception as e:
    #    print(f"Failed to store predicted activities: {e}")
    #    raise
    return predicted_activity_df
def evaluate_predictions(predicted_activity_df,exp_activity_df,round_activity_df,num_var,round_num,evolve_directory):
    try:
        predict = predicted_activity_df["actual_activity"].isna()
        extract_predict = predicted_activity_df[predict]
        #print(extract_predict)
        top_var = extract_predict.iloc[0:num_var]
        #print(top_var)
        top_var_real = pd.merge(top_var,exp_activity_df,on='seq_id', how='inner')
        top_var_real = top_var_real[['seq_id','activity']]
        #top_var_csv_path = evolve_directory / f"Round_{round_num}_top_variants.xlsx"
        #top_var_real.to_excel(top_var_csv_path, index=False)
        next_round_activity = pd.concat([round_activity_df,top_var_real], ignore_index=True)
    except Exception as e:
        print(f"Failed to Evaluate Predictions")
        raise
    return next_round_activity


    return combined_df
def evolve_simulation(wt_aa_seq,embeddings_path,exp_activity_file_path,num_var,round_num):
    exp_activity_df = pd.read_excel(exp_activity_file_path)
    raw_embedding_df = pd.read_csv(embeddings_path)
    raw_activity_df = exp_activity_df.sample(num_var)
    while not raw_activity_df['seq_id'].isin(raw_embedding_df['seq_id']).all():
        raw_activity_df = exp_activity_df.sample(num_var)
    print(raw_activity_df)
    evolve_directory = Path("evolve") / Path(exp_activity_file_path).stem
    evolve_directory.mkdir(parents=True, exist_ok=True)
    exp_data = [raw_activity_df.sort_values('activity',ascending=True)]
    for i in range(1,(round_num+1)):
        if i == 1:
            predicted_activity_df = train_model(wt_aa_seq,raw_activity_df,raw_embedding_df,evolve_directory)
            current_round_activity_df = evaluate_predictions(predicted_activity_df,exp_activity_df,raw_activity_df,num_var,i,evolve_directory)
            #print(i)
            #print(current_round_activity_df)
            
        else:
            predicted_activity_df = train_model(wt_aa_seq,current_round_activity_df,raw_embedding_df,evolve_directory)
            current_round_activity_df = evaluate_predictions(predicted_activity_df,exp_activity_df,current_round_activity_df,num_var,i,evolve_directory)
        exp_data.append(current_round_activity_df.sort_values('activity',ascending=True))      
    print(exp_data)
    
    

In [156]:
wt_aa_seq = 'MAKEDNIEMQGTVLETLPNTMFRVELENGHVVTAHISGKMRKNYIRILTGDKVTVELTPYDLSKGRIVFRSR'
activity_file_path = r'E:\ProgrammingProjects\foldy\backend\EvolveTest\kelsic_Round1.xlsx'
exp_activity_file_path = r'E:\ProgrammingProjects\foldy\backend\EvolveTest\Kelsic_Data_foldy.xlsx'
embeddings_path = r'E:\ProgrammingProjects\foldy\backend\EvolveTest\007426_embeddings_esmc_600m_wt_dms.csv'
num_var = 12
round_num = 3
evolve_simulation(wt_aa_seq,embeddings_path,exp_activity_file_path,num_var,round_num)

     seq_id  activity
390    N28M  0.849000
437     A2V  1.032750
126    T16H  0.970500
1383    I7E  0.930000
397    N28V  0.848000
421     A2C  0.893500
926    K52H  0.963000
717    K42V  0.880000
1158   L62W  0.948000
1109   Y60L  0.956167
754    Y44R  0.927000
507    T33I  1.016000
(1369, 1152)
(1369, 1152)
(1369, 1152)
[     seq_id  activity
397    N28V  0.848000
390    N28M  0.849000
717    K42V  0.880000
421     A2C  0.893500
754    Y44R  0.927000
1383    I7E  0.930000
1158   L62W  0.948000
1109   Y60L  0.956167
926    K52H  0.963000
126    T16H  0.970500
507    T33I  1.016000
437     A2V  1.032750,    seq_id  activity
12   T12I  0.826667
4    N28V  0.848000
0    N28M  0.849000
7    K42V  0.880000
5     A2C  0.893500
15    E8I  0.908667
14   T49V  0.916000
16   T49I  0.926000
10   Y44R  0.927000
3     I7E  0.930000
20   T54I  0.945000
8    L62W  0.948000
9    Y60L  0.956167
21   Q10I  0.959000
6    K52H  0.963000
2    T16H  0.970500
23   T33L  0.999167
18   T33V  1.008750
17    A